# N2-01b Structure audit: UNSW IoT attack dataset (Hamza et al., SOSR 2019)
Purpose: N2 needs identity AND attacks AND routing. UNSW-IoTraffic (01) has no attacks. This dataset has benign + attack traces for ~10 of the same lab devices.
STRUCTURE ONLY: file lists, headers, counts. No model, no label-vs-feature analysis. Output decides whether this dataset can carry N2 (declared criteria below).

In [ ]:
import os, io, json, glob, zipfile, shutil, requests
import pandas as pd
OUT = '/kaggle/working/n2_01b_attack_audit'; os.makedirs(OUT, exist_ok=True)
DATA = '/kaggle/working/unsw_attack'; os.makedirs(DATA, exist_ok=True)
BASE = 'https://iotanalytics.unsw.edu.au/anomaly-data/'
FILES = ['flowdata.zip', 'annotations.zip', 'attackinfo.xlsx']
# declared before inspecting the data:
CRITERIA = dict(min_devices=8, min_benign_rows_per_device=5000, min_attack_instances_per_device=30,
                min_attack_types_overall=4, needs_timestamps=True)
print(CRITERIA)

In [ ]:
def find_local(n):
    h = glob.glob(f'/kaggle/input/**/{n}', recursive=True); return h[0] if h else None
paths, fails = {}, {}
for n in FILES:
    try:
        p = find_local(n)
        if not p:
            p = os.path.join(DATA, n)
            with requests.get(BASE + n, stream=True, timeout=600, headers={'User-Agent': 'ris-n2-audit'}) as r:
                r.raise_for_status()
                with open(p, 'wb') as fh: shutil.copyfileobj(r.raw, fh)
        paths[n] = p; print('OK  ', n, os.path.getsize(p))
    except Exception as e:
        fails[n] = repr(e); print('FAIL', n, e)
json.dump({'paths': paths, 'fails': fails}, open(f'{OUT}/download_log.json', 'w'), indent=1)
# Kaggle auto-extracts uploaded zips: also accept extracted folders
extra = [d for d in glob.glob('/kaggle/input/*/*') if os.path.isdir(d)]
print('extracted input dirs:', extra[:10])
assert paths or extra, 'Nothing available: download flowdata.zip, annotations.zip, attackinfo.xlsx in a browser and attach as a Kaggle dataset'

## Archive structure

In [ ]:
struct = {}
for n in ['flowdata.zip', 'annotations.zip']:
    if n in paths and zipfile.is_zipfile(paths[n]):
        with zipfile.ZipFile(paths[n]) as z:
            infos = z.infolist()
            struct[n] = [(i.filename, i.file_size) for i in infos]
            print('=====', n, len(infos), 'entries, total', sum(i.file_size for i in infos)//10**6, 'MB')
            for i in infos[:60]: print(f'  {i.file_size:>12}  {i.filename}')
            z.extractall(f'{DATA}/{n[:-4]}')
json.dump(struct, open(f'{OUT}/archive_structure.json', 'w'), indent=1)
if not struct:   # extracted dirs from Kaggle input
    for d in extra:
        fl = [os.path.relpath(f, d) for f in glob.glob(d + '/**/*', recursive=True) if os.path.isfile(f)]
        print('=====', d, len(fl)); print(fl[:60])

## attackinfo.xlsx (device and attack info)

In [ ]:
xp = paths.get('attackinfo.xlsx') or find_local('attackinfo.xlsx')
if xp:
    x = pd.read_excel(xp, sheet_name=None)
    for sh, df in x.items():
        print('=== sheet', sh, df.shape); print(df.head(40).to_string())
        df.to_csv(f"{OUT}/attackinfo_{sh}.csv", index=False)

## Headers and first rows of every data file (no analysis)

In [ ]:
roots = [f'{DATA}/flowdata', f'{DATA}/annotations'] + extra
seen = []
for r in roots:
    for f in sorted(glob.glob(r + '/**/*', recursive=True)):
        if os.path.isfile(f) and f.lower().endswith(('.csv', '.txt', '.json')):
            seen.append(f)
print(len(seen), 'readable data files')
for f in seen[:25]:
    try:
        d = pd.read_csv(f, nrows=3, low_memory=False)
        print('---', os.path.relpath(f, '/kaggle'), os.path.getsize(f)//1024, 'KB'); print(list(d.columns)[:40]); print(d.head(2).to_string()[:800])
    except Exception as e:
        print('---', f, 'unreadable', e)
json.dump(seen, open(f'{OUT}/data_files.json', 'w'), indent=1)

## Cheap counts: rows and time span per data file (for the declared criteria)

In [ ]:
rows = []
for f in seen:
    try:
        d = pd.read_csv(f, low_memory=False)
    except Exception:
        continue
    tc = next((c for c in d.columns if any(h in str(c).lower() for h in ['time', 'ts', 'stamp', 'date'])), None)
    rec = {'file': os.path.relpath(f, '/kaggle'), 'rows': len(d), 'cols': d.shape[1], 'time_col': tc}
    if tc is not None:
        s = pd.to_numeric(d[tc], errors='coerce')
        rec.update(tmin=s.min(), tmax=s.max())
    txt = [c for c in d.columns if d[c].dtype == object and d[c].nunique() <= 40]
    rec['low_card_text_cols'] = txt[:6]
    rows.append(rec)
tab = pd.DataFrame(rows); tab.to_csv(f'{OUT}/file_counts.csv', index=False)
print(tab.to_string(index=False)[:12000])

In [ ]:
shutil.make_archive('/kaggle/working/n2_01b_attack_audit_bundle', 'zip', OUT)
print(sorted(os.listdir(OUT)))